In [15]:
MAIN_PROMPTS_FILE = "/kaggle/working/data/prompts.txt"         
EMOTION_PROMPTS_FILE = "/kaggle/working/data/emotion_prompts.txt" 

!mkdir "/kaggle/working/data"
with open(MAIN_PROMPTS_FILE, 'w') as f:
    f.write("""
a person wearing a space helmet
a person as a superhero
a person in a business suit holding a coffee
a person as a chef cooking pasta
a person skiing down a mountain
a person playing electric guitar on stage
a person as a pirate with a parrot
a person meditating in lotus pose
a person as an astronaut getting ready for the flight
a person riding a bicycle in the park
a person with a birthday cake
a person as a detective with a magnifying glass
a person as a knight in armor
a person surfing a big wave
a person as a painter at an easel
a person reading a book under a tree
a person as a wizard with a magic wand
a person taking a selfie with a dog
a person as a scientist with test tubes
a person dancing at a disco party
""")

with open(EMOTION_PROMPTS_FILE, 'w') as f:
    f.write("""
a person smiling happily
a person crying with tears
a person looking very angry
a person with a surprised expression
a person looking sad and thoughtful
a person laughing out loud
a person with a scared expression
a person looking disgusted
""")

# ========== ТЕКСТОВЫЕ ЛЕЙБЛЫ ДЛЯ CLIP-КЛАССИФИКАТОРА ЭМОЦИЙ ==========
# EMOTION_LABELS = [
#     "a happy smiling face",
#     "a sad crying face", 
#     "an angry furious face",
#     "a surprised shocked face",
#     "a neutral expressionless face",
# ]

In [18]:
import torch
import os
import numpy as np
import pandas as pd
from PIL import Image
from diffusers import StableDiffusionPipeline, DDIMScheduler
from peft import PeftModel
from insightface.app import FaceAnalysis
import open_clip
import torch.nn as nn
from tqdm import tqdm

# ========== НАСТРОЙКИ ==========
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16

LORA_PATH = "/kaggle/input/datasets/kolyagudkov/lora-diplom-weights"
REFERENCES_DIR = "/kaggle/input/datasets/kolyagudkov/ref2diplom"
MAIN_PROMPTS_FILE = "/kaggle/working/data/prompts.txt"         # подсунешь
EMOTION_PROMPTS_FILE = "/kaggle/working/data/emotion_prompts.txt"  # подсунешь

RESULTS_DIR = "/kaggle/working/results_all"
METRICS_OUTPUT = "/kaggle/working/metrics_all.csv"
SUMMARY_OUTPUT = "/kaggle/working/metrics_all_summary.csv"

os.makedirs(RESULTS_DIR, exist_ok=True)

# ========== КОНФИГУРАЦИИ ==========
# Все 6 комбинаций
CONFIGS = [
    {"name": "Base",               "use_lora": False, "use_ip": False, "adapter_type": None},
    {"name": "Base_LoRA",          "use_lora": True,  "use_ip": False, "adapter_type": None},
    {"name": "IP_Face",            "use_lora": False, "use_ip": True,  "adapter_type": "face"},
    {"name": "IP_Plus",            "use_lora": False, "use_ip": True,  "adapter_type": "plus"},
    {"name": "IP_Face_LoRA",       "use_lora": True,  "use_ip": True,  "adapter_type": "face"},
    {"name": "IP_Plus_LoRA",       "use_lora": True,  "use_ip": True,  "adapter_type": "plus"},
]

# ========== ЧАСТЬ 1: ГЕНЕРАЦИЯ ==========

def load_pipeline(config):
    """Загружает SD1.5 + опционально LoRA и IP-Adapter."""
    print(f"  Загрузка SD1.5...")
    pipe = StableDiffusionPipeline.from_pretrained(
        "SG161222/Realistic_Vision_V4.0_noVAE",
        torch_dtype=DTYPE,
        use_safetensors=True,
    )
    pipe = pipe.to(DEVICE)
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

    if config["use_lora"]:
        print(f"  Загрузка LoRA...")
        pipe.unet = PeftModel.from_pretrained(pipe.unet, LORA_PATH)
        pipe.unet.set_adapter("default")

    if config["use_ip"]:
        print(f"  Загрузка IP-Adapter ({config['adapter_type']})...")
        if config["adapter_type"] == "face":
            weight_name = "ip-adapter-plus-face_sd15.safetensors"
        else:
            weight_name = "ip-adapter-plus_sd15.safetensors"

        pipe.load_ip_adapter(
            "h94/IP-Adapter",
            subfolder="models",
            weight_name=weight_name,
            image_encoder_folder="models/image_encoder",
        )
        pipe.set_ip_adapter_scale(0.7)

    return pipe


def generate_all():
    """Генерация для всех конфигураций и всех наборов промптов."""
    # Загружаем промпты
    with open(MAIN_PROMPTS_FILE, "r") as f:
        main_prompts = [l.strip() for l in f if l.strip()]

    with open(EMOTION_PROMPTS_FILE, "r") as f:
        emotion_prompts = [l.strip() for l in f if l.strip()]

    # Загружаем референсы
    ref_files = sorted([f for f in os.listdir(REFERENCES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    ref_images = {}
    for f in ref_files:
        name = os.path.splitext(f)[0]
        ref_images[name] = Image.open(os.path.join(REFERENCES_DIR, f)).convert("RGB")

    # Два набора промптов
    prompt_sets = {
        "main": main_prompts,
        "emotion": emotion_prompts,
    }

    for config in CONFIGS:
        print(f"\n{'='*50}")
        print(f"Конфигурация: {config['name']}")
        print(f"{'='*50}")

        # Проверяем, есть ли уже генерация
        config_dir = os.path.join(RESULTS_DIR, config["name"])
        existing = len(os.listdir(config_dir)) if os.path.isdir(config_dir) else 0
        expected = len(ref_images) * (len(main_prompts) + len(emotion_prompts))

        if existing >= expected:
            print(f"  [SKIP] Уже сгенерировано {existing}/{expected}")
            continue

        os.makedirs(config_dir, exist_ok=True)
        pipe = load_pipeline(config)

        for set_name, prompts in prompt_sets.items():
            for person_name, ref_image in ref_images.items():
                # Если нет IP-Adapter — референс не нужен для генерации, 
                # но для метрик он понадобится
                for i, prompt in enumerate(prompts):
                    filename = f"{person_name}_{set_name}_{i:02d}.png"
                    filepath = os.path.join(config_dir, filename)

                    if os.path.exists(filepath):
                        continue

                    try:
                        generator = torch.Generator(device=DEVICE).manual_seed(42)
                        kwargs = {
                            "prompt": prompt,
                            "negative_prompt": "low quality, blurry, distorted, ugly, bad anatomy",
                            "num_inference_steps": 30,
                            "guidance_scale": 7.0,
                            "generator": generator,
                            "height": 512,
                            "width": 512,
                        }
                        if config["use_ip"]:
                            kwargs["ip_adapter_image"] = ref_image

                        result = pipe(**kwargs)
                        result.images[0].save(filepath)
                    except Exception as e:
                        print(f"  [ERROR] {filename}: {e}")

        del pipe
        torch.cuda.empty_cache()

    print("\n✅ Генерация завершена!")


# ========== ЧАСТЬ 2: МЕТРИКИ ==========

def load_metric_models():
    """Загружает модели для подсчёта метрик."""
    print("Загрузка CLIP...")
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="laion2b_s34b_b79k"
    )
    clip_model = clip_model.to(DEVICE).eval()
    clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

    print("Загрузка InsightFace...")
    face_app = FaceAnalysis(
        name="buffalo_l",
        providers=["CUDAExecutionProvider" if DEVICE == "cuda" else "CPUExecutionProvider"],
    )
    face_app.prepare(ctx_id=0 if DEVICE == "cuda" else -1)

    print("Загрузка Aesthetic Predictor...")
    aesthetic_clip, _, aesthetic_preprocess = open_clip.create_model_and_transforms(
        "ViT-L-14", pretrained="openai"
    )
    aesthetic_clip = aesthetic_clip.to(DEVICE).eval()

    aesthetic_model_path = "/kaggle/working/aesthetic_predictor.pth"
    if not os.path.exists(aesthetic_model_path):
        import urllib.request
        urllib.request.urlretrieve(
            "https://github.com/christophschuhmann/improved-aesthetic-predictor/raw/main/sac%2Blogos%2Bava1-l14-linearMSE.pth",
            aesthetic_model_path,
        )

    state_dict = torch.load(aesthetic_model_path, map_location=DEVICE)

    class AestheticMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleDict({})
            self.layers["0"] = nn.Linear(768, 1024)
            self.layers["2"] = nn.Linear(1024, 128)
            self.layers["4"] = nn.Linear(128, 64)
            self.layers["6"] = nn.Linear(64, 16)
            self.layers["7"] = nn.Linear(16, 1)
        def forward(self, x):
            x = nn.functional.relu(self.layers["0"](x))
            x = nn.functional.relu(self.layers["2"](x))
            x = nn.functional.relu(self.layers["4"](x))
            x = nn.functional.relu(self.layers["6"](x))
            x = self.layers["7"](x)
            return x.squeeze(-1)

    aesthetic_mlp = AestheticMLP().to(DEVICE)
    aesthetic_mlp.load_state_dict(state_dict, strict=True)
    aesthetic_mlp.eval()

    return clip_model, clip_preprocess, clip_tokenizer, face_app, aesthetic_clip, aesthetic_preprocess, aesthetic_mlp


def compute_all_metrics():
    """Считает метрики для всех конфигураций."""
    clip_model, clip_preprocess, clip_tokenizer, face_app, aesthetic_clip, aesthetic_preprocess, aesthetic_mlp = load_metric_models()

    # Загружаем все промпты (для CLIP-T)
    with open(MAIN_PROMPTS_FILE, "r") as f:
        main_prompts = [l.strip() for l in f if l.strip()]
    with open(EMOTION_PROMPTS_FILE, "r") as f:
        emotion_prompts = [l.strip() for l in f if l.strip()]
    all_prompts = main_prompts + emotion_prompts

    ref_files = sorted([f for f in os.listdir(REFERENCES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))])

    # Кэш для CLIP-T (чтобы не кодировать текст каждый раз)
    text_emb_cache = {}
    for prompt in all_prompts:
        text = clip_tokenizer([prompt]).to(DEVICE)
        with torch.no_grad():
            text_emb_cache[prompt] = clip_model.encode_text(text)
            text_emb_cache[prompt] = text_emb_cache[prompt] / text_emb_cache[prompt].norm(dim=-1, keepdim=True)

    results = []

    # Считаем общее количество
    total = 0
    for config in CONFIGS:
        config_path = os.path.join(RESULTS_DIR, config["name"])
        if os.path.isdir(config_path):
            total += len([f for f in os.listdir(config_path) if f.endswith(".png")])

    pbar = tqdm(total=total, desc="Метрики")

    for config in CONFIGS:
        config_path = os.path.join(RESULTS_DIR, config["name"])
        if not os.path.isdir(config_path):
            continue

        gen_files = sorted([f for f in os.listdir(config_path) if f.endswith(".png")])

        for gen_file in gen_files:
            # Имя: person_main_03.png или person_emotion_01.png
            parts = gen_file.replace(".png", "").split("_")
            person_name = parts[0]
            set_type = parts[1]  # "main" или "emotion"
            prompt_idx = int(parts[2])

            if set_type == "main":
                prompt_text = main_prompts[prompt_idx]
            else:
                prompt_text = emotion_prompts[prompt_idx]

            # Референс
            ref_path = None
            for ref_file in ref_files:
                if os.path.splitext(ref_file)[0] == person_name:
                    ref_path = os.path.join(REFERENCES_DIR, ref_file)
                    break

            if ref_path is None:
                pbar.update(1)
                continue

            gen_path = os.path.join(config_path, gen_file)

            # CLIP-I
            ref_img = clip_preprocess(Image.open(ref_path).convert("RGB")).unsqueeze(0).to(DEVICE)
            gen_img = clip_preprocess(Image.open(gen_path).convert("RGB")).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                ref_emb = clip_model.encode_image(ref_img)
                gen_emb = clip_model.encode_image(gen_img)
                ref_emb = ref_emb / ref_emb.norm(dim=-1, keepdim=True)
                gen_emb = gen_emb / gen_emb.norm(dim=-1, keepdim=True)
                clip_i = float((ref_emb * gen_emb).sum(dim=-1).item())
                clip_t = float((gen_emb * text_emb_cache[prompt_text]).sum(dim=-1).item())

            # Face Similarity
            ref_faces = face_app.get(np.array(Image.open(ref_path).convert("RGB")))
            gen_faces = face_app.get(np.array(Image.open(gen_path).convert("RGB")))
            if len(ref_faces) > 0 and len(gen_faces) > 0:
                ref_face = max(ref_faces, key=lambda x: (x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]))
                gen_face = max(gen_faces, key=lambda x: (x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]))
                face_sim = float(np.dot(ref_face.normed_embedding, gen_face.normed_embedding))
            else:
                face_sim = float("nan")

            # Aesthetic
            aest_img = aesthetic_preprocess(Image.open(gen_path).convert("RGB")).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                features = aesthetic_clip.encode_image(aest_img)
                features = features / features.norm(dim=-1, keepdim=True)
                aesthetic = float(aesthetic_mlp(features).item())

            results.append({
                "config": config["name"],
                "person": person_name,
                "set": set_type,
                "prompt": prompt_text,
                "CLIP_I": round(clip_i, 4),
                "CLIP_T": round(clip_t, 4),
                "Face_Similarity": round(face_sim, 4) if not np.isnan(face_sim) else "N/A",
                "Aesthetic_Score": round(aesthetic, 2),
            })

            pbar.update(1)

    pbar.close()

    df = pd.DataFrame(results)
    df.to_csv(METRICS_OUTPUT, index=False)

    # ========== СВОДНАЯ ТАБЛИЦА ==========
    df_clean = df.replace("N/A", float("nan"))
    df_clean["Face_Similarity"] = pd.to_numeric(df_clean["Face_Similarity"], errors="coerce")

    # Общая таблица
    summary_all = (
        df_clean.groupby("config")
        .agg(
            CLIP_I=("CLIP_I", "mean"),
            CLIP_T=("CLIP_T", "mean"),
            Face_Sim=("Face_Similarity", "mean"),
            Aesthetic=("Aesthetic_Score", "mean"),
            Total=("CLIP_I", "count"),
            Face_Detected=("Face_Similarity", lambda x: x.notna().sum()),
        )
        .round(4)
    )
    summary_all["Face_Det%"] = (summary_all["Face_Detected"] / summary_all["Total"] * 100).round(1)

    # Раздельно по сетам
    for set_name, set_label in [("main", "Основные промпты"), ("emotion", "Промпты эмоций")]:
        subset = df_clean[df_clean["set"] == set_name]
        summary_set = (
            subset.groupby("config")
            .agg(
                CLIP_I=("CLIP_I", "mean"),
                CLIP_T=("CLIP_T", "mean"),
                Face_Sim=("Face_Similarity", "mean"),
                Aesthetic=("Aesthetic_Score", "mean"),
            )
            .round(4)
        )

        print(f"\n{'='*80}")
        print(f"📊 {set_label.upper()}")
        print(f"{'='*80}")
        print(f"{'Конфигурация':<20} {'CLIP-I':>8} {'CLIP-T':>8} {'Face Sim':>8} {'Aesthetic':>8}")
        print("-"*80)
        for cfg in CONFIGS:
            name = cfg["name"]
            if name in summary_set.index:
                row = summary_set.loc[name]
                print(f"{name:<20} {row['CLIP_I']:.3f}   {row['CLIP_T']:.3f}   {row['Face_Sim']:.3f}   {row['Aesthetic']:.2f}")

    summary_all.to_csv(SUMMARY_OUTPUT)
    print(f"\n✅ Детально: {METRICS_OUTPUT}")
    print(f"✅ Сводка: {SUMMARY_OUTPUT}")


# ========== MAIN ==========
if __name__ == "__main__":
    generate_all()
    compute_all_metrics()


Конфигурация: Base
  Загрузка SD1.5...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--SG161222--Realistic_Vision_V4.0_noVAE/snapshots/1685907c0283c7278ba26c5fe561506f564b48d3/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/30 [00:00<?, ?it/s]

...

100%|██████████| 281857/281857 [00:02<00:00, 109406.66KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
Загрузка Aes

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
Метрики: 100%|██████████| 840/840 [15:23<00:00,  1.10s/it]
/tmp/ipykernel_57/1206134085.py:319: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean = df.replace("N/A", float("nan"))



📊 ОСНОВНЫЕ ПРОМПТЫ
Конфигурация           CLIP-I   CLIP-T Face Sim Aesthetic
--------------------------------------------------------------------------------
Base                 0.262   0.325   0.014   4.06
Base_LoRA            0.287   0.261   -0.003   3.79
IP_Face              0.526   0.302   0.226   4.06
IP_Plus              0.864   0.127   0.236   4.06
IP_Face_LoRA         0.452   0.211   0.137   3.79
IP_Plus_LoRA         0.632   0.112   0.157   3.82

📊 ПРОМПТЫ ЭМОЦИЙ
Конфигурация           CLIP-I   CLIP-T Face Sim Aesthetic
--------------------------------------------------------------------------------
Base                 0.426   0.332   -0.021   3.99
Base_LoRA            0.335   0.235   nan   3.75
IP_Face              0.765   0.255   0.245   4.02
IP_Plus              0.846   0.252   0.199   4.06
IP_Face_LoRA         0.470   0.221   0.105   3.78
IP_Plus_LoRA         0.532   0.227   0.117   3.78

✅ Детально: /kaggle/working/metrics_all.csv
✅ Сводка: /kaggle/working/metrics_all_s

In [ ]:
# Просто печатаем что-то каждые 30 секунд
import time

while True:
    print(".", end="", flush=True)
    time.sleep(30)

......................................................................................................................................................................................................................................